# Agent的高级用法
## 1.设置agent的名称
使用name字段进行设置
举例：

In [ ]:
from langchain.agents import create_agent
import os

from langchain.agents.structured_output import ProviderStrategy
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # 关键修改：关闭思考模式
    # extra_body={
    #     "thinking": {
    #         "type": "disabled"
    #     }
    # },
)
agent=create_agent(
    model=model,
    name="agent01"
)
response=agent.invoke({
    "messages":[
        {"role":"system","content":"你是一个精通数学的老师，擅长以通俗易懂的方式讲解数学问题"},
        {"role":"user","content":"100+20*3=？"}
    ]
})
#rprint(response)
for message in response["messages"]:
    message.pretty_print()

## 2.系统提示词的设置
使用system_prompt参数进行设置，可以是str，也可以是SystemMessage
举例1：

In [ ]:
from langchain_tavily import TavilySearch
from langchain.agents import create_agent
import os
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # 关键修改：关闭思考模式
    # extra_body={
    #     "thinking": {
    #         "type": "disabled"
    #     }
    # },
)
# 2.导入工具
web_search = TavilySearch(max_results=2)
# 3.创建Agent
agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="你是一名多才多艺的智能助手，可以调用工具帮助用户解决问题。"
)
# 4.运行Agent获得结果
result = agent.invoke(
    {"messages": [
        {"role": "user", "content": "请帮我查询2026年足球世界杯是哪个国家举办的？"}
    ]}
)
#print(result['messages'][-1].content)
rprint(result)

举例2：

In [ ]:
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint

# 工具：实现两数相加
@tool
def add_numbers(a: int, b: int) -> str:
    """计算并返回两个数的和。"""
    return f"和为：{a + b}"

# 创建客服助手Agent
agent = create_agent(
    model=model,
    tools=[add_numbers],  # 工具列表
    # system_prompt="你是一个数学助手，解决日常的算术问题"
    system_prompt=SystemMessage(content="你是一个数学助手，解决日常的算术问题")
)

response = agent.invoke(
    {"messages": [
        {"role": "user", "content": "10加上20再加上30是多少？"}
    ]},
)
rprint(response)
# print(response["messages"][-1].content)同上

举例3：

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import SystemMessage, HumanMessage

flag = 0

@tool
def get_weather(city: str):
    """
    天气查询工具
    Args:
    city: 城市名称
    """
    global flag
    flag += 1
    if flag < 3:
        # raise Exception("暂时无法访问") return "TEMP_UNAVAILABLE: 天气服务暂时不可用，请稍后重试"
        return f"{city}今天天气挺好"

messages = [
    HumanMessage("你好，杭州今天的天气如何？")
]

agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt=SystemMessage(
        "你是一个天气助手。"
        "当工具返回以 'TEMP_UNAVAILABLE:' 开头的结果时，"
        "说明是临时故障，不要立即放弃；"
        "你应再次调用同一个工具，最多重试 3 次。"
        "如果 3 次后仍失败，再向用户说明服务暂时不可用。"
    )
)

response = agent.invoke({"messages": messages})
# print(response)
for msg in response["messages"]:
    msg.pretty_print()

## 3.结构化输出的4种策略
### 3.1ProviderStrategy策略(ds不支持)

In [ ]:

from pydantic import BaseModel,Field
import os
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # #关键修改：关闭思考模式
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)
#使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name:str=Field(description="用户的姓名")
    email:str=Field(description="用户的邮箱")
    phone:str=Field(description="用户的电话")
agent=create_agent(
    model=model,
    response_format=ProviderStrategy(ContractInfo)
)
response=agent.invoke({
    "messages": [
        {
            "role": "user","content":"从以下信息中提取用户信息，小明的邮箱是shkstar@163.com,电话是1345751122"
        }]
})
rprint(response)

## 3.2ToolStrategy策略

In [ ]:

from langchain.agents.structured_output import ToolStrategy
from pydantic import BaseModel,Field
import os
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # #关键修改：关闭思考模式
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)
#使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name:str=Field(description="用户的姓名")
    email:str=Field(description="用户的邮箱")
    phone:str=Field(description="用户的电话")
agent=create_agent(
    model=model,
    response_format=ToolStrategy(ContractInfo)
)
response=agent.invoke({
    "messages": [
        # {
        #     "role": "user","content":"从以下信息中提取用户信息，小明的邮箱是shkstar@163.com,电话是1345751122"
        # }
        HumanMessage("从以下信息中提取用户信息，小明的邮箱是shkstar@163.com,电话是1345751122")
    ]
})
#rprint(response)
print(response["structured_response"])

### 3.3 AutoStrategry/Type策略

In [ ]:

from langchain.agents.structured_output import ToolStrategy, AutoStrategy
from pydantic import BaseModel,Field
import os
from langchain_deepseek import ChatDeepSeek
from dotenv import load_dotenv
from scripts.regsetup import description
from rich import print as rprint
#1.读取.env配置文件信息,相关的环境变量以.env文件中的优先
load_dotenv(verbose=True)
DEEPSEEK_API_KEY=os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL=os.getenv("DEEPSEEK_BASE_URL")
#2.模型初始化
model=ChatDeepSeek(
    model="deepseek-v4-flash",
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    # #关键修改：关闭思考模式
    extra_body={
        "thinking": {
            "type": "disabled"
        }
    },
)
#使用Pydantic结构化方式定义
class ContractInfo(BaseModel):
    """用户的联系方式"""
    name:str=Field(description="用户的姓名")
    email:str=Field(description="用户的邮箱")
    phone:str=Field(description="用户的电话")
agent=create_agent(
    model=model,
    response_format=AutoStrategy(ContractInfo)
)
response=agent.invoke({
    "messages": [
        # {
        #     "role": "user","content":"从以下信息中提取用户信息，小明的邮箱是shkstar@163.com,电话是1345751122"
        # }
        HumanMessage("从以下信息中提取用户信息，小明的邮箱是shkstar@163.com,电话是1345751122")
    ]
})
rprint(response)
#print(response["structured_response"])